## 0. Configurando sessão spark

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("bronze_to_silver_aluno")
    .config(
        "spark.jars.packages",
        "com.google.cloud.spark:spark-bigquery-with-dependencies_2.13:0.44.2"
    )
    .getOrCreate()
)

# Projeto usado para faturamento das consultas
spark.conf.set("parentProject", "tech-challenge-fase-2-505123")

:: loading settings :: url = jar:file:/opt/micromamba/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/jupyter/.ivy2.5.2/cache
The jars for the packages stored in: /home/jupyter/.ivy2.5.2/jars
com.google.cloud.spark#spark-bigquery-with-dependencies_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-e4b91111-5b7a-4b50-b59a-7c4aa10b5efa;1.0
	confs: [default]
	found com.google.cloud.spark#spark-bigquery-with-dependencies_2.13;0.44.2 in central
:: resolution report :: resolve 177ms :: artifacts dl 4ms
	:: modules in use:
	com.google.cloud.spark#spark-bigquery-with-dependencies_2.13;0.44.2 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	-----------------------------------------

In [2]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 20)
spark.conf.set("spark.sql.repl.eagerEval.truncate", 100)

## 1. Imports

In [3]:
from pyspark.sql import functions as F

## 2. Geração de parâmetros

In [4]:
par_source_project = "tech-challenge-fase-2-505123"
par_source_silver_municipio = f"{par_source_project}.silver.municipio"
par_source_silver_aluno = f"{par_source_project}.silver.aluno"
par_source_silver_alfabetizacao_municipio = f"{par_source_project}.silver.meta_alfabetizacao_municipio"
par_source_diretorio_brasil = f"basedosdados.br_bd_diretorios_brasil.municipio"

par_source_gold_aluno = f"{par_source_project}.gold.dim_municipio"

## 3. Leitura dos dados da origem

In [5]:
df_scr_municipio = spark.read.format("bigquery").option("table",par_source_silver_municipio).load()
df_scr_alfabetizacao_municipio = spark.read.format("bigquery").option("table",par_source_silver_alfabetizacao_municipio).load()
df_scr_aluno = spark.read.format("bigquery").option("table",par_source_silver_aluno).load()
dir_mun = (spark.read.format("bigquery").option("table", par_source_diretorio_brasil).load())

## 4. Transformações

### 4.1. id_municipio de todas as tabelas

In [6]:
universo_mun = (
    df_scr_municipio.select("id_municipio")
    .union(df_scr_aluno.select("id_municipio"))
    .union(df_scr_alfabetizacao_municipio.select("id_municipio"))
    .distinct()
)

### 4.2. Cruzando com dados do IBGE

In [7]:
dim_municipio = (
    universo_mun
    .join(
        dir_mun.select("id_municipio","nome","sigla_uf","nome_uf","nome_regiao"),
        on= "id_municipio",
        how= "left"
    )
    .withColumnRenamed("nome","nome_municipio")
)

## 5. Armazenamento no BQ

In [9]:
(
    dim_municipio.write.format("bigquery")
    .option("table", par_source_gold_aluno)
    .option("writeMethod", "direct")
    .mode("overwrite")
    .save()
)